# Run a trained walking policy and measure what the motors have to deliver

This notebook loads one of the `standwalk` PPO policies, runs it in MuJoCo through a short command schedule
(stand → walk → walk and turn → stand), and produces:

1. a **video** with a panel in the top-right corner showing every motor's torque in Nm and as a ratio of its peak (limit);
2. a **torque time series** for every motor over the whole run (at the 250 Hz physics rate), with the limits;
3. a **motor-sizing table**: peak and RMS torque, utilisation, time near the limit, peak mechanical power.

**Colab:** any runtime works (a GPU only makes rendering faster). Just run all cells: the second cell clones the
[`metal2humanoid`](https://github.com/hbrnetwork/metal2humanoid) course repository, which contains this project, the
trained checkpoints and the robot models, and moves into `projects/standwalk`.

**Locally:** run it from `projects/standwalk` in your clone of the repository.

In [ ]:
import os, sys, subprocess, pathlib, glob
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mujoco>=3.1", "imageio", "imageio-ffmpeg", "pandas"], check=True)
    has_gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    if has_gpu:   # let glvnd find the NVIDIA EGL driver (as in the official MuJoCo Colab)
        icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
        if not os.path.exists(icd):
            os.makedirs(os.path.dirname(icd), exist_ok=True)
            open(icd, "w").write('{ "file_format_version" : "1.0.0", "ICD" : { "library_path" : "libEGL_nvidia.so.0" } }')
        os.environ["MUJOCO_GL"] = "egl"
    else:         # CPU runtime: software rendering
        subprocess.run("apt-get install -y -qq libosmesa6 > /dev/null 2>&1", shell=True)
        os.environ["MUJOCO_GL"] = "osmesa"; os.environ["PYOPENGL_PLATFORM"] = "osmesa"
    print("Colab runtime, GPU:", has_gpu, "| MUJOCO_GL =", os.environ["MUJOCO_GL"])
elif sys.platform.startswith("linux") and not os.environ.get("DISPLAY"):
    os.environ.setdefault("MUJOCO_GL", "egl")

In [ ]:
# --- code, checkpoints and robot models: all from the course repository ---
REPO = "https://github.com/hbrnetwork/metal2humanoid.git"
if not pathlib.Path("standwalk").is_dir():                 # not running inside projects/standwalk (e.g. Colab): get the course repo
    if not pathlib.Path("metal2humanoid").is_dir():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO], check=True)
    os.chdir("metal2humanoid/projects/standwalk")           # code, checkpoints/ and ../../embodiments all live here
if "." not in sys.path: sys.path.insert(0, ".")
available = sorted(glob.glob("checkpoints/*.pt")) or sorted(glob.glob("runs/*_walk/best.pt"))
print("project folder:", os.getcwd(), "\navailable checkpoints:", available)


## 1. Choose the policy

Each checkpoint is self-describing: it knows its embodiment, joint order, default pose, PD gains, torque limits and
observation layout, so picking the file is all that is needed.

In [ ]:
CHECKPOINT = "checkpoints/asimov_walk.pt"          # <- change to any file listed above, e.g. checkpoints/bhl_walk.pt
if not os.path.exists(CHECKPOINT):
    CHECKPOINT = available[0]
import warnings; warnings.filterwarnings("ignore")
import numpy as np, torch, mujoco, imageio.v2 as imageio, matplotlib.pyplot as plt, pandas as pd
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Video, HTML
import standwalk
from standwalk.env import HumanoidVecEnv, TaskConfig
from standwalk.ppo import Policy
from standwalk.render import EnvRenderer, tracking_camera

policy = Policy(CHECKPOINT)
ck = policy.ckpt
EMB, TASK = ck["embodiment"], ck["task_config"]["task"]
joint_names, tau_max = ck["joint_names"], np.array(ck["torque_limit"])
print(f"{CHECKPOINT}: embodiment={EMB} task={TASK} trained {ck['meta'].get('total_steps', 0):,} steps")
print(f"{len(joint_names)} motors, control period {ck['ctrl_dt']*1000:.0f} ms, action scale {ck['action_scale']} rad")
print(pd.DataFrame({"motor": joint_names, "kp": ck["kp"], "kd": ck["kd"], "torque limit [Nm]": tau_max}).to_string(index=False))

## 2. Run the policy with a command schedule and record everything

The robot gets a sequence of velocity commands. At every control step (20 ms) we store the torques of all motors at each of
the 5 physics substeps (4 ms), the joint velocities, foot contacts and the base velocity. Frames are rendered at 25 fps with the
torque panel drawn on top.

In [ ]:
SCHEDULE = [(2.0, [0.0, 0.0, 0.0]),    # seconds, [vx m/s, vy m/s, yaw rate rad/s]
            (6.0, [0.5, 0.0, 0.0]),
            (4.0, [0.3, 0.0, 0.6]),
            (3.0, [0.0, 0.0, 0.0])]
FRAME_EVERY = 2            # render every 2nd control step -> 25 fps
W, H = 800, 600

def short(name):
    s = name.replace("_joint", "").replace("leg_", "").replace("arm_", "")
    return s.replace("left_", "L ").replace("right_", "R ").replace("_", " ")

try:
    FONT = ImageFont.load_default(size=12); FONT_S = ImageFont.load_default(size=11)
except TypeError:
    FONT = FONT_S = ImageFont.load_default()

def draw_panel(frame, tau, t, cmd):
    im = Image.fromarray(frame); d = ImageDraw.Draw(im, "RGBA")
    n = len(tau); row = 14; pw = 300; ph = 40 + n * row
    x0, y0 = im.size[0] - pw - 8, 8
    d.rectangle([x0, y0, x0 + pw, y0 + ph], fill=(0, 0, 0, 160))
    d.text((x0 + 8, y0 + 5), f"t = {t:5.2f} s   cmd vx {cmd[0]:+.1f}  vy {cmd[1]:+.1f}  yaw {cmd[2]:+.1f}", fill="white", font=FONT)
    d.text((x0 + 8, y0 + 21), "motor         torque / limit", fill=(190, 190, 190), font=FONT_S)
    bx, bw = x0 + 112, 90
    for i in range(n):
        y = y0 + 38 + i * row
        r = min(abs(tau[i]) / tau_max[i], 1.0)
        col = (int(255 * min(1, 2 * r)), int(255 * min(1, 2 * (1 - r))), 50, 230)
        d.text((x0 + 8, y), short(joint_names[i])[:16], fill="white", font=FONT_S)
        d.rectangle([bx, y + 2, bx + bw, y + row - 3], outline=(140, 140, 140, 200))
        d.rectangle([bx, y + 2, bx + int(bw * r), y + row - 3], fill=col)
        d.text((bx + bw + 6, y), f"{tau[i]:+6.1f} Nm {100*abs(tau[i])/tau_max[i]:4.0f}%", fill="white", font=FONT_S)
    return np.asarray(im)

task = TaskConfig(**ck["task_config"])
env = HumanoidVecEnv(EMB, num_envs=1, task=task, seed=3, n_threads=1)
dt, dts, dec = env.dt, env.le.emb.sim_dt, env.decimation
try:
    renderer = EnvRenderer(env, W, H, camera=tracking_camera(env, azimuth=120, elevation=-15))
    renderer.frame(0); can_render = True
except Exception as e:
    print("rendering unavailable (", e, ") - the torque analysis still runs"); can_render = False

log = {"t": [], "tau": [], "qd": [], "contact": [], "cmd": [], "v": []}
frames, t, fell = [], 0.0, False
obs, _ = env.reset()
for seconds, cmd in SCHEDULE:
    env.set_commands(cmd, fixed=True)
    for k in range(int(round(seconds / dt))):
        obs, _, rew, term, trunc, info = env.step(policy(obs))
        tau_sub, qd_sub = env.substep_torques[0].copy(), env.substep_joint_vel[0].copy()
        log["t"].append(t + dts * np.arange(1, dec + 1)); log["tau"].append(tau_sub); log["qd"].append(qd_sub)
        log["contact"].append(env.foot_contact[0].copy()); log["cmd"].append(np.array(cmd)); log["v"].append(env.base_lin_vel[0].copy())
        if can_render and k % FRAME_EVERY == 0:
            peak = tau_sub[np.argmax(np.abs(tau_sub), axis=0), np.arange(tau_sub.shape[1])]   # signed peak within this control step
            frames.append(draw_panel(renderer.frame(0), peak, t, cmd))
        t += dt
        if term[0]:
            fell = True; print(f"the robot fell at t = {t:.2f} s"); break
    if fell: break
env.close()
T = np.concatenate(log["t"]); TAU = np.concatenate(log["tau"]); QD = np.concatenate(log["qd"])
CONTACT = np.array(log["contact"]); CMD = np.array(log["cmd"]); V = np.array(log["v"]); TC = np.arange(len(CMD)) * dt
print(f"ran {t:.1f} s, {len(T)} torque samples per motor, {len(frames)} frames, fell={fell}")

## 3. Video with the torque panel

In [ ]:
if frames:
    out = f"{EMB}_{TASK}_torques.mp4"
    imageio.mimwrite(out, frames, fps=int(round(1 / dt / FRAME_EVERY)), codec="libx264", quality=6, macro_block_size=None)
    print("wrote", out, f"{os.path.getsize(out)/1e6:.1f} MB")
    display(Video(out, embed=True, width=720))

## 4. Torque time series for every motor

Dashed lines are the torque limits from the registry (what the PD controller saturates at). The grey strips show when each
foot is in contact; the band at the top shows the commanded forward speed. Torques are the actual PD outputs at the 4 ms
physics rate, so short peaks at touchdown are visible.

In [ ]:
n = len(joint_names); ncols = 3; nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows + 1, ncols, figsize=(15, 1.7 * (nrows + 1)), sharex=True, gridspec_kw={"height_ratios": [0.8] + [1] * nrows})
for ax in axes[0, 1:]: ax.axis("off")
ax = axes[0, 0]; ax.step(TC, CMD[:, 0], where="post", label="cmd vx [m/s]"); ax.plot(TC, V[:, 0], lw=0.8, label="actual vx")
ax.step(TC, CMD[:, 2], where="post", label="cmd yaw [rad/s]", c="C2"); ax.legend(fontsize=7, ncol=3, loc="upper right"); ax.set_ylabel("command")
for i in range(n):
    ax = axes[1 + i // ncols, i % ncols]
    for f, c in ((0, "0.85"), (1, "0.92")):
        on = CONTACT[:, f]
        for s, e in zip(np.where(np.diff(np.r_[0, on]) == 1)[0], np.where(np.diff(np.r_[on, 0]) == -1)[0]):
            ax.axvspan(TC[s], TC[e] + dt, color=c, lw=0, zorder=0)
    ax.plot(T, TAU[:, i], lw=0.7, color="C0" if "left" in joint_names[i] else "C3")
    ax.axhline(tau_max[i], ls="--", c="k", lw=0.6); ax.axhline(-tau_max[i], ls="--", c="k", lw=0.6)
    ax.set_title(joint_names[i], fontsize=8); ax.set_ylabel("Nm", fontsize=7); ax.tick_params(labelsize=7)
for j in range(n, nrows * ncols): axes[1 + j // ncols, j % ncols].axis("off")
for ax in axes[-1]: ax.set_xlabel("time [s]")
fig.suptitle(f"{EMB} / {TASK}: motor torques (grey = left/right foot in contact)", y=1.0); plt.tight_layout(); plt.show()

In [ ]:
# utilisation heat-map: |torque| / limit for every motor over time
util = np.abs(TAU) / tau_max
fig, ax = plt.subplots(figsize=(15, 0.32 * n + 1.2))
im = ax.imshow(util.T, aspect="auto", cmap="magma", vmin=0, vmax=1, extent=[T[0], T[-1], n - 0.5, -0.5], interpolation="nearest")
ax.set_yticks(range(n)); ax.set_yticklabels([short(j) for j in joint_names], fontsize=8); ax.set_xlabel("time [s]")
plt.colorbar(im, ax=ax, pad=0.01, label="|torque| / limit"); ax.set_title("motor utilisation"); plt.tight_layout(); plt.show()

## 5. Motor-sizing summary

What each motor would need to deliver to execute this policy: peak and RMS torque, how often it runs near its limit, and the
peak mechanical power $|\tau \cdot \dot q|$. A motor running >80 % of its limit for a sizeable fraction of the time is a
candidate for a bigger actuator (or for a lower PD gain / stronger regularisation in training).

In [ ]:
power = TAU * QD
rows = []
for i, j in enumerate(joint_names):
    a = np.abs(TAU[:, i])
    rows.append({"motor": j, "limit [Nm]": tau_max[i], "peak |tau| [Nm]": a.max(), "peak / limit": a.max() / tau_max[i],
                 "RMS tau [Nm]": np.sqrt(np.mean(TAU[:, i] ** 2)), "time > 50% limit": np.mean(a > 0.5 * tau_max[i]),
                 "time > 80% limit": np.mean(a > 0.8 * tau_max[i]), "peak |power| [W]": np.abs(power[:, i]).max(),
                 "mean |power| [W]": np.abs(power[:, i]).mean()})
summary = pd.DataFrame(rows)
display(summary.round(3).style.format({"peak / limit": "{:.0%}", "time > 50% limit": "{:.0%}", "time > 80% limit": "{:.0%}"})
        .background_gradient(subset=["peak / limit"], cmap="Reds", vmin=0, vmax=1))
csv = f"{EMB}_{TASK}_torques.csv"
pd.DataFrame(TAU, columns=joint_names).assign(t=T).set_index("t").to_csv(csv)
print("torque time series saved to", csv, "| mean |power| over all motors: %.1f W" % np.abs(power).mean(axis=1).mean(),
      "| peak total |power|: %.1f W" % np.abs(power).sum(axis=1).max())
if IN_COLAB:
    from google.colab import files
    for f in ([out] if frames else []) + [csv]:
        files.download(f)

### Reading the numbers

* The torques come from the PD law $\tau = k_p(q^* - q) - k_d\dot q$ saturated at the limit, so they depend on the gains in the
  checkpoint as much as on the gait. Softer gains give lower peaks and a less precise gait.
* The limits in the registry are plausible guesses, not the datasheet values of your actuators. Replace them (and the gains)
  with the real motor-driver settings in `standwalk/embodiments.py` and retrain before trusting absolute numbers.
* Touchdown peaks are short (tens of ms). A motor driver's peak-torque rating covers those; the RMS column is what matters
  for thermal limits.